# arc3-v22-aa — INSTRUMENT CONTROL A/A: stock vs stock (phase-order drift) on the 27B V22 serving stack (one boot, two 25-game phases: stock_a -> stock_b)

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/taaf-duck-qwen38-serving-v1", "driessmit1/arc3-vllm-h100-wheelhouse-v3"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")


# V22 only modifies the vLLM launch flags inside the bundled setup command.
# The target model owns native MTP tensors and the original checkpoint validation
# already verifies that mtp.* tensors are mounted.
def _v31_serving_commands(command: str) -> tuple[str, str, bool]:
    if "vllm.entrypoints.openai.api_server" not in command:
        return command, command, False

    source_block = """        '--kv-cache-dtype',
        'fp8',
    ]"""

    v22_block = """        '--kv-cache-dtype',
        'fp8',
        '--speculative-config',
        '{"method":"mtp","num_speculative_tokens":3}',
        '--async-scheduling',
    ]"""

    v31_block = """        '--kv-cache-dtype',
        'fp8',
        '--speculative-config',
        '{"method":"mtp","num_speculative_tokens":3}',
        '--async-scheduling',
        '--no-enable-chunked-prefill',
    ]"""

    if source_block not in command:
        print("V31: known source serving block not found; leaving it unchanged.", flush=True)
        return command, command, False

    return (
        command.replace(source_block, v31_block, 1),
        command.replace(source_block, v22_block, 1),
        True,
    )


def _v22_cleanup_partial_server() -> None:
    import signal

    pid_path = WORKING_DIR / "vllm-openai-server.pid"
    if not pid_path.exists():
        return
    try:
        pid = int(pid_path.read_text(encoding="utf-8").strip())
        try:
            os.kill(pid, signal.SIGTERM)
            time.sleep(2)
        except OSError:
            pass
        try:
            os.kill(pid, 0)
        except OSError:
            pass
        else:
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
    except Exception as exc:
        print(f"V22: partial-server cleanup warning: {exc!r}", flush=True)
    finally:
        pid_path.unlink(missing_ok=True)


# Solver setup commands run before the benchmark loads.
# Primary = exact V22 MTP3 stack + no-chunked-prefill.
# Fallback = exact MTP3+async serving that produced 2.66 LB.
env = _command_env()
for original_command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    primary_command, v22_command, optimized = _v31_serving_commands(original_command)

    if not optimized:
        print(f"taaf.kaggle: setup command: {original_command}", flush=True)
        subprocess.run(original_command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    else:
        print("V31: launching MTP3 + async + FP8 KV + no-chunked-prefill.", flush=True)
        try:
            subprocess.run(primary_command, shell=True, check=True, cwd=WORKING_DIR, env=env)
        except subprocess.CalledProcessError as exc:
            print(
                f"V31: primary startup failed ({exc!r}); "
                "falling back to exact V22 MTP3+async.",
                flush=True,
            )
            _v22_cleanup_partial_server()
            subprocess.run(v22_command, shell=True, check=True, cwd=WORKING_DIR, env=env)

    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# ---- V31 runtime resilience ----
import threading as _v31_threading
import urllib.request as _v31_urllib

_V31_URL = os.environ.get("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1").rstrip("/")
_V31_PID = WORKING_DIR / "vllm-openai-server.pid"
_V31_LOG = WORKING_DIR / "vllm-openai-server.log"
_V31_STOP = _v31_threading.Event()
_V31_THREAD = None
_V31_LOCK = _v31_threading.Lock()


def _v31_healthy(timeout: float = 4.0) -> bool:
    try:
        with _v31_urllib.urlopen(f"{_V31_URL}/models", timeout=timeout) as response:
            return 200 <= int(response.status) < 500
    except Exception:
        return False


def _v31_wait_healthy(timeout: float = 480.0) -> bool:
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if _v31_healthy(5.0):
            return True
        time.sleep(5)
    return False


def _v31_kill() -> None:
    if not _V31_PID.exists():
        return
    try:
        pid = int(_V31_PID.read_text(encoding="utf-8").strip())
        try:
            os.kill(pid, signal.SIGTERM)
            time.sleep(2)
        except OSError:
            pass
        try:
            os.kill(pid, 0)
        except OSError:
            pass
        else:
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
    except Exception as exc:
        print(f"V31 cleanup warning: {exc!r}", flush=True)
    finally:
        _V31_PID.unlink(missing_ok=True)


def _v31_spawn(no_chunked: bool) -> None:
    provenance_file = WORKING_DIR / "qwen38-model-provenance.json"
    provenance = json.loads(provenance_file.read_text(encoding="utf-8"))
    model_path = str(provenance["model_path"])

    site_packages = WORKING_DIR / "vllm-site-packages"
    child_env = os.environ.copy()
    current_pp = child_env.get("PYTHONPATH", "")
    if str(site_packages) not in current_pp.split(os.pathsep):
        child_env["PYTHONPATH"] = (
            str(site_packages)
            if not current_pp
            else str(site_packages) + os.pathsep + current_pp
        )
    child_env.update({
        "USE_TF": "0",
        "TRANSFORMERS_NO_TF": "1",
        "TRANSFORMERS_NO_TORCHVISION": "1",
        "VLLM_NO_USAGE_STATS": "1",
    })

    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model_path,
        "--served-model-name", os.environ.get("INFERENCE_ANALYZER_MODEL", "Qwen/Qwen3.8-27B-FP8"),
        "--host", "127.0.0.1",
        "--port", "1234",
        "--tensor-parallel-size", "1",
        "--enable-auto-tool-choice",
        "--tool-call-parser", "qwen3_coder",
        "--generation-config", "vllm",
        "--enable-prefix-caching",
        "--default-chat-template-kwargs", '{"preserve_thinking": true}',
        "--reasoning-parser", "qwen3",
        "--max-model-len", "262144",
        "--kv-cache-dtype", "fp8",
        "--speculative-config", '{"method":"mtp","num_speculative_tokens":3}',
        "--async-scheduling",
    ]
    if no_chunked:
        cmd.append("--no-enable-chunked-prefill")

    # Prevent unbounded log growth across restarts.
    if _V31_LOG.exists():
        previous = WORKING_DIR / "vllm-openai-server.previous.log"
        try:
            previous.unlink(missing_ok=True)
            _V31_LOG.replace(previous)
        except OSError:
            pass

    log_handle = _V31_LOG.open("w", encoding="utf-8")
    process = subprocess.Popen(
        cmd,
        env=child_env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
    _V31_PID.write_text(str(process.pid), encoding="utf-8")

    if not _v31_wait_healthy():
        raise RuntimeError("restarted vLLM did not become healthy in time")


def _v31_recover() -> bool:
    with _V31_LOCK:
        if _v31_healthy():
            return True

        print("V31 watchdog: recovering MTP3/no-chunk server.", flush=True)
        _v31_kill()
        try:
            _v31_spawn(no_chunked=True)
            print("V31 watchdog: primary recovery succeeded.", flush=True)
            return True
        except Exception as first:
            print(f"V31 watchdog: primary recovery failed: {first!r}", flush=True)

        _v31_kill()
        try:
            _v31_spawn(no_chunked=False)
            print("V31 watchdog: exact V22 MTP3 recovery succeeded.", flush=True)
            return True
        except Exception as second:
            print(f"V31 watchdog: fallback recovery failed: {second!r}", flush=True)
            _v31_kill()
            return False


def _v31_start_watchdog(solver) -> None:
    global _V31_THREAD
    _V31_STOP.clear()

    def worker():
        failures = 0
        while not _V31_STOP.wait(15):
            if _v31_healthy():
                failures = 0
                continue

            failures += 1
            print(f"V31 watchdog: health failure {failures}/3", flush=True)
            if failures < 3:
                continue

            if _v31_recover():
                failures = 0
                continue

            print(
                "V31 watchdog: server unrecoverable; stopping solver to preserve partial score.",
                flush=True,
            )
            stop_event = getattr(solver, "_stop_event", None)
            if stop_event is not None:
                stop_event.set()
            return

    _V31_THREAD = _v31_threading.Thread(
        target=worker,
        name="v31-watchdog",
        daemon=True,
    )
    _V31_THREAD.start()


def _v31_stop_watchdog() -> None:
    _V31_STOP.set()
    if _V31_THREAD is not None:
        _V31_THREAD.join(timeout=5)


if not _v31_healthy(10):
    raise RuntimeError("V31 preflight: local vLLM API is not healthy.")
print("V31 preflight: local vLLM API healthy.", flush=True)



In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# V31 keeps the V22 gameplay/prompt/tool policy unchanged.
bm.solver.save_request_logs = False
print("V31 concurrency:", getattr(bm.solver, "concurrency", None))
print("V31 analyzer_timeout:", getattr(bm.solver, "analyzer_timeout", None))
print("V31 save_request_logs:", getattr(bm.solver, "save_request_logs", None))


In [ ]:
# ==== graft install (A/B machinery; every flag phase-controlled) ====
# All grafts install ONCE; behaviour is env-gated per phase. Defaults ALL OFF —
# the stock phase must be byte-equivalent stock behaviour.
import importlib as _il

for _flag in ['TP_ENABLE', 'TP2_ENABLE', 'TP4_ENABLE', 'TP5_ENABLE', 'TP6_ENABLE', 'TP7_ENABLE', 'TP8_ENABLE', 'TP9_ENABLE', 'TP10_ENABLE']:
    os.environ[_flag] = "0"

_G_DIR = WORKING_DIR / "graft_bundle"
_G_DIR.mkdir(parents=True, exist_ok=True)
_GRAFT_SOURCES = {'graft_pipeline.py': '"""Turn-pipeline repair graft (TP9, 2026-08-31) — no discarded thinking,\nlivelock breaker, one client-timeout retry.\n\nEvidence (docs/research-2026-08-31/R9-path-to-7-gap-analysis.md §2.1): 48% of\nall LLM turns are idle (`step_executed: False`), overwhelmingly "Yielded\ncontrol to solver: turn_time_budget"; 13 vLLM read-timeout turns stranded\ngames at run end; r11l sat in a 6,000 s deterministic livelock (20+\nconsecutive idle turns with byte-identical transcript lengths) after clearing\nL1 in 5 actions.\n\nSeams, verified in the anim bundle (the code that plays at eval —\nsubmission/_inspect_replay/assets_build/ARC3-Inference):\n\n- THE YIELD. tool_agent.py:154 reads LOCAL_ANALYZER_YIELD_SECONDS (60 in the\n  measured run) into :1085 `self._yield_seconds`; `control_yield_reason`\n  (:2154-2163) returns "turn_time_budget" once the turn is past it. The check\n  fires between requests (:2168 loop top, :2288 after a no-tool-call reply,\n  :2353 between tool calls) and breaks out of `analyze`.\n\n- WHERE THE THINKING DIES. On a `requests.RequestException` (read-timeout\n  included) `analyze` sets preserve_history=False (:2365), reverts\n  `_history_messages` to the pre-turn snapshot (:2403), and returns\n  `AnalyzerTurnResult(step_executed=False, retryable_failure=True,\n  reasoning=captured_reasoning)` (:2380) — the ONLY surviving copy of the\n  turn\'s reasoning is `result.reasoning`. The solver loop (solver.py:352-363)\n  handles retryable_failure / yielded_control / idle by `continue` and NEVER\n  reads `result.reasoning`: that field is dropped on the floor every idle\n  turn. (On a clean turn_time_budget yield after a no-tool-call reply the\n  reasoning-only assistant message does land in `_history_messages`, but the\n  next `analyze` rebuilds context from a trimmed history and — at fixed\n  seed/temperature — regenerates the same doomed turn; r11l\'s identical\n  transcript lengths are exactly that. So even the "kept" case needs the\n  explicit resume-and-act nudge, and the kept copy can be trimmed away.)\n\n- THE HTTP CALL. `_chat_completion` (:1518-1546) does one `requests.post`\n  with timeout=`request_timeout_seconds` (solver.py:267-284: min of analyzer\n  timeout, wall remaining, soft remaining). A ReadTimeout surfaces as\n  `requests.RequestException` -> the discard path above; the solver retries\n  the analysis step after a 1 s backoff (solver.py:64,352-357) but the\n  generation is already lost, and at run end `should_stop()` breaks first\n  (solver.py:354-355) — the 13 stranded turns.\n\nThree behaviours, each individually gated (TP9_ENABLE=0 turns all off):\n\n(a) TURN PERSIST/RESUME (TP9_RESUME, default 1). Wrap `ToolAgent.analyze`:\n    when the stock turn comes back idle (yielded_control or\n    retryable_failure) with non-empty `result.reasoning`, stash the tail\n    (TP9_RESUME_CHARS, default 1500) on the agent. Wrap\n    `ToolAgent._build_user_prompt`: the next turn\'s user prompt gets a\n    RESUME block carrying that tail plus an instruction to continue from it\n    and act — consumed once, cleared on any executed step, reset on game\n    change (mirrors _ensure_session\'s runtime-dir keying, :1140-1152).\n    J9/F6: a consumed tail is parked, not dropped — if the very request it\n    was injected into dies with NO new reasoning (timeout before any reply),\n    the parked tail is restored and re-injected next turn; any new reasoning\n    or an executed step discards the parked copy.\n\n(b) LIVELOCK DETECTOR (TP9_LIVELOCK, default 1; TP9_LIVELOCK_K, default 3;\n    TP9_LIVELOCK_COOLDOWN, default 5). Per game, hash each idle turn\'s\n    `result.reasoning` (empty output hashes equal — that IS the r11l\n    signature). K consecutive identical hashes with zero executed actions\n    arms a LOOP BREAKER block for the next user prompt ("state ONE new\n    hypothesis and emit one action([...]) batch now") and bumps a counter.\n    Streak resets on an executed step or a differing hash.\n    J9/F3a: `retryable_failure` turns are EXCLUDED from the streak — they\n    carry no model-produced output (a 3 s server outage at the solver\'s 1 s\n    retry cadence must never arm a false "identical output" accusation);\n    they neither build nor reset the streak.\n    J9/F3b: the breaker fires AT MOST ONCE per armed streak — injection\n    resets the streak and starts a cooldown of TP9_LIVELOCK_COOLDOWN counted\n    turns during which no streak accumulates, bounding the injection rate to\n    1 per (K + cooldown) turns (~13 per 100-turn persistent livelock at\n    defaults, vs every prompt before the fix).\n    Scope (J9/F5): hash identity only catches the empty/deterministic-replay\n    subspecies; a livelock that still emits sampled non-identical text at\n    temp 0.6 is out of scope for this detector.\n\n(c) CLIENT-TIMEOUT RETRY (TP9_RETRY, default 1; TP9_RETRY_BACKOFF, default\n    1.0 s; TP9_RETRY_MIN_BUDGET, default 5.0 s). Wrap `_chat_completion`:\n    retry `requests.Timeout` / `requests.ConnectionError` up to TP9_RETRY\n    times before letting the exception reach the discard path. Other\n    RequestExceptions (HTTP errors, context-length rejections handled at\n    :2221) pass through untouched — a context-overflow retry of the\n    identical payload deterministically fails and the analyze loop has its\n    own recovery for it.\n    J9/F4 wall-clock guard: the turn\'s `request_timeout_seconds` is the\n    solver\'s min(analyzer timeout, wall remaining, soft remaining)\n    (solver.py:267-284), so it IS the wall bound. The retry never exceeds\n    it: the retry attempt gets only the leftover budget\n    (budget - elapsed - backoff), and when that leftover is under\n    TP9_RETRY_MIN_BUDGET the retry is skipped and the exception propagates\n    (stock behaviour, letting the solver\'s own should_stop-guarded retry\n    path take over). Net effect: a fast ConnectionError still gets its\n    near-free retry; a ReadTimeout that consumed the whole budget is never\n    doubled. With no budget argument and no analyzer timeout the retry is\n    unbounded, as before.\n\nConventions: module-level _STOCK dict (never only a closure), fail-open\ntry/except around all graft logic, rebinding class attributes only, new file\nonly. install() -> "pipeline: OK" / "pipeline: SKIP (...)".\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport os\nimport time\nfrom typing import Any\n\n_STATE: dict[str, Any] = {\n    "installed": False,\n    "resumes_injected": 0,\n    "perturbations_injected": 0,\n    "retries_used": 0,\n}\n_STOCK: dict[str, Any] = {}\n_OFF = {"0", "false", "no", "off"}\n\nDEFAULT_LIVELOCK_K = 3\nDEFAULT_LIVELOCK_COOLDOWN = 5\nDEFAULT_RETRY = 1\nDEFAULT_RESUME_CHARS = 1500\nDEFAULT_RETRY_BACKOFF = 1.0\nDEFAULT_RETRY_MIN_BUDGET = 5.0\n\nRESUME_BLOCK = (\n    "RESUME: You were interrupted mid-thought last turn ({reason}) and nothing was applied. "\n    "Your partial reasoning from that turn (tail):\\n<<<\\n{tail}\\n>>>\\n"\n    "Do not re-derive this from scratch. Continue from those conclusions and emit a `python` "\n    "tool call that reaches `action(actions)` promptly this turn."\n)\nPERTURB_BLOCK = (\n    "LOOP BREAKER: You have produced identical output {n} times in a row with no actions "\n    "executed. Break the loop: state ONE new hypothesis in a single sentence, then emit one "\n    "`python` tool call that calls `action([...])` with a small batch NOW — act first, "\n    "analyze the result after."\n)\n\n\n# ----------------------------------------------------------------- flags ---\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef enabled() -> bool:\n    return _env("TP9_ENABLE", "1").lower() not in _OFF\n\n\ndef resume_enabled() -> bool:\n    return enabled() and _env("TP9_RESUME", "1").lower() not in _OFF\n\n\ndef livelock_enabled() -> bool:\n    return enabled() and _env("TP9_LIVELOCK", "1").lower() not in _OFF\n\n\ndef livelock_k() -> int:\n    try:\n        return max(2, int(_env("TP9_LIVELOCK_K", str(DEFAULT_LIVELOCK_K))))\n    except ValueError:\n        return DEFAULT_LIVELOCK_K\n\n\ndef livelock_cooldown() -> int:\n    try:\n        return max(0, int(_env("TP9_LIVELOCK_COOLDOWN", str(DEFAULT_LIVELOCK_COOLDOWN))))\n    except ValueError:\n        return DEFAULT_LIVELOCK_COOLDOWN\n\n\ndef retry_attempts() -> int:\n    if not enabled():\n        return 0\n    try:\n        return max(0, int(_env("TP9_RETRY", str(DEFAULT_RETRY))))\n    except ValueError:\n        return DEFAULT_RETRY\n\n\ndef retry_backoff() -> float:\n    try:\n        return max(0.0, float(_env("TP9_RETRY_BACKOFF", str(DEFAULT_RETRY_BACKOFF))))\n    except ValueError:\n        return DEFAULT_RETRY_BACKOFF\n\n\ndef retry_min_budget() -> float:\n    try:\n        return max(0.0, float(_env("TP9_RETRY_MIN_BUDGET", str(DEFAULT_RETRY_MIN_BUDGET))))\n    except ValueError:\n        return DEFAULT_RETRY_MIN_BUDGET\n\n\ndef resume_chars() -> int:\n    try:\n        return max(200, int(_env("TP9_RESUME_CHARS", str(DEFAULT_RESUME_CHARS))))\n    except ValueError:\n        return DEFAULT_RESUME_CHARS\n\n\ndef status() -> dict[str, Any]:\n    return {\n        "installed": _STATE["installed"],\n        "enabled": enabled(),\n        "resume": resume_enabled(),\n        "livelock": livelock_enabled(),\n        "livelock_k": livelock_k(),\n        "livelock_cooldown": livelock_cooldown(),\n        "retry": retry_attempts(),\n        "retry_min_budget": retry_min_budget(),\n        "resume_chars": resume_chars(),\n        "resumes_injected": _STATE["resumes_injected"],\n        "perturbations_injected": _STATE["perturbations_injected"],\n        "retries_used": _STATE["retries_used"],\n    }\n\n\n# ----------------------------------------------------------- per-game state ---\nclass PipelineState:\n    def __init__(self) -> None:\n        self.runtime_dir: Any = None\n        self.resume_tail: str | None = None\n        self.resume_reason: str = ""\n        # F6: the last-injected tail, parked so it can be restored if the\n        # request it rode on dies with no new reasoning.\n        self.injected_tail: str | None = None\n        self.injected_reason: str = ""\n        self.last_hash: str | None = None\n        self.idle_streak = 0\n        self.perturb_pending = False\n        # F3b: counted turns left before the livelock streak may build again.\n        self.cooldown = 0\n\n\ndef _pstate(agent: Any, state_path: Any = None) -> PipelineState:\n    """Get (or create) the agent\'s TP9 state; reset it when the game changes.\n\n    Mirrors ToolAgent._ensure_session, which keys a session by\n    state_path.parent (the per-game runtime dir).\n    """\n    st = getattr(agent, "_tp9", None)\n    if st is None:\n        st = PipelineState()\n        try:\n            agent._tp9 = st\n        except Exception:  # noqa: BLE001\n            pass\n    if state_path is not None:\n        try:\n            runtime_dir = state_path.parent\n            if st.runtime_dir is not None and st.runtime_dir != runtime_dir:\n                fresh = PipelineState()\n                fresh.runtime_dir = runtime_dir\n                try:\n                    agent._tp9 = fresh\n                except Exception:  # noqa: BLE001\n                    pass\n                return fresh\n            st.runtime_dir = runtime_dir\n        except Exception:  # noqa: BLE001\n            pass\n    return st\n\n\ndef _idle_reason(result: Any) -> str:\n    if getattr(result, "retryable_failure", False):\n        return "the model-server request failed"\n    if getattr(result, "yielded_control", False):\n        return "the turn time budget expired"\n    return "no action call was captured"\n\n\ndef _note_turn_result(st: PipelineState, result: Any) -> None:\n    """Update resume/livelock state from one stock analyze() result."""\n    if getattr(result, "step_executed", False):\n        st.resume_tail = None\n        st.resume_reason = ""\n        st.injected_tail = None\n        st.injected_reason = ""\n        st.last_hash = None\n        st.idle_streak = 0\n        st.perturb_pending = False\n        st.cooldown = 0\n        return\n    reasoning = str(getattr(result, "reasoning", "") or "")\n    if resume_enabled():\n        if reasoning.strip():\n            st.resume_tail = reasoning[-resume_chars():]\n            st.resume_reason = _idle_reason(result)\n            st.injected_tail = None\n            st.injected_reason = ""\n        elif st.injected_tail and getattr(result, "retryable_failure", False):\n            # F6 (re-judged): restore the parked tail ONLY when the request it\n            # rode on actually died (retryable_failure). A clean empty yield\n            # means the model saw the nudge and ignored it — re-serving there\n            # stacks duplicate RESUME blocks in a livelock (J9 regression iii).\n            st.resume_tail = st.injected_tail\n            st.resume_reason = st.injected_reason\n    if livelock_enabled():\n        if getattr(result, "retryable_failure", False):\n            # F3a: a failed request produced no model output — a server\n            # outage must neither build nor reset the identical-output streak.\n            return\n        if st.cooldown > 0:\n            # F3b: cooling down after an injection; no streak accumulation.\n            st.cooldown -= 1\n            st.last_hash = None\n            st.idle_streak = 0\n            return\n        digest = hashlib.sha1(reasoning.encode("utf-8", errors="replace")).hexdigest()\n        if digest == st.last_hash:\n            st.idle_streak += 1\n        else:\n            st.idle_streak = 1\n            st.last_hash = digest\n        if st.idle_streak >= livelock_k():\n            st.perturb_pending = True\n\n\n# --------------------------------------------------------------- install ---\ndef install() -> str:\n    if _STATE["installed"]:\n        return "pipeline: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"pipeline: SKIP (import failed: {exc!r})"\n    try:\n        import requests\n    except Exception as exc:  # noqa: BLE001\n        return f"pipeline: SKIP (requests missing: {exc!r})"\n    cls = getattr(agent_mod, "ToolAgent", None)\n    if cls is None:\n        return "pipeline: SKIP (missing ToolAgent)"\n    for name in ("analyze", "_build_user_prompt", "_chat_completion"):\n        if getattr(cls, name, None) is None:\n            return f"pipeline: SKIP (ToolAgent.{name} missing)"\n\n    _STOCK["analyze"] = cls.analyze\n    _STOCK["build_user_prompt"] = cls._build_user_prompt\n    _STOCK["chat_completion"] = cls._chat_completion\n\n    # -- (a)+(b) capture: wrap analyze -------------------------------------\n    def analyze(self, state_path, action_num, *args, **kwargs):\n        if enabled():\n            try:\n                # Reset per-game state BEFORE the turn so a stale resume tail\n                # from the previous game never leaks into this game\'s prompt.\n                _pstate(self, state_path)\n            except Exception:  # noqa: BLE001\n                pass\n        result = _STOCK["analyze"](self, state_path, action_num, *args, **kwargs)\n        if not enabled() or result is None:\n            return result\n        try:\n            _note_turn_result(_pstate(self, state_path), result)\n        except Exception:  # noqa: BLE001\n            pass\n        return result\n\n    # -- (a)+(b) injection: wrap _build_user_prompt ------------------------\n    def build_user_prompt(self, action_num, **kwargs):\n        text = _STOCK["build_user_prompt"](self, action_num, **kwargs)\n        if not enabled():\n            return text\n        try:\n            st = getattr(self, "_tp9", None)\n            if st is None:\n                return text\n            blocks: list[str] = []\n            if resume_enabled() and st.resume_tail:\n                blocks.append(RESUME_BLOCK.format(reason=st.resume_reason, tail=st.resume_tail))\n                # consumed once — but parked (F6) so a dead injected request\n                # can restore it; discarded on new reasoning or executed step.\n                st.injected_tail = st.resume_tail\n                st.injected_reason = st.resume_reason\n                st.resume_tail = None\n                st.resume_reason = ""\n                _STATE["resumes_injected"] += 1\n            if livelock_enabled() and st.perturb_pending:\n                blocks.append(PERTURB_BLOCK.format(n=st.idle_streak))\n                st.perturb_pending = False\n                # F3b: at most one injection per armed streak — reset the\n                # streak and start the re-arm cooldown.\n                st.idle_streak = 0\n                st.last_hash = None\n                st.cooldown = livelock_cooldown()\n                _STATE["perturbations_injected"] += 1\n            if blocks:\n                return text + "\\n" + "\\n".join(blocks)\n        except Exception:  # noqa: BLE001\n            pass\n        return text\n\n    # -- (c) retry: wrap _chat_completion ----------------------------------\n    def chat_completion(self, messages, **kwargs):\n        attempts = 0\n        budget = None\n        started = None\n        try:\n            attempts = retry_attempts()\n            # F4: the turn\'s request_timeout_seconds is the solver\'s wall\n            # bound (min of analyzer timeout / wall remaining / soft\n            # remaining) — the retry must stay inside it.\n            budget = kwargs.get("request_timeout_seconds")\n            if budget is None:\n                budget = getattr(self, "_timeout", None)\n            budget = None if budget is None else float(budget)\n            started = time.monotonic()\n        except Exception:  # noqa: BLE001\n            attempts = 0\n        if attempts <= 0:\n            return _STOCK["chat_completion"](self, messages, **kwargs)\n        for attempt in range(attempts + 1):\n            try:\n                return _STOCK["chat_completion"](self, messages, **kwargs)\n            except (requests.Timeout, requests.ConnectionError):\n                if attempt >= attempts:\n                    raise\n                try:\n                    backoff = retry_backoff()\n                    if budget is not None:\n                        leftover = budget - (time.monotonic() - started) - backoff\n                        if leftover < retry_min_budget():\n                            # F4: no meaningful wall left — propagate (stock\n                            # behaviour; the solver\'s should_stop-guarded\n                            # retry path takes over).\n                            raise\n                        kwargs = dict(kwargs)\n                        kwargs["request_timeout_seconds"] = leftover\n                    _STATE["retries_used"] += 1\n                    if backoff > 0:\n                        time.sleep(backoff)\n                except (requests.Timeout, requests.ConnectionError):\n                    raise\n                except Exception:  # noqa: BLE001\n                    pass\n        raise RuntimeError("unreachable: TP9 retry loop exhausted without raising")\n\n    analyze._tp9_stock = _STOCK["analyze"]\n    build_user_prompt._tp9_stock = _STOCK["build_user_prompt"]\n    chat_completion._tp9_stock = _STOCK["chat_completion"]\n    cls.analyze = analyze\n    cls._build_user_prompt = build_user_prompt\n    cls._chat_completion = chat_completion\n    _STATE["installed"] = True\n    return "pipeline: OK"\n'}
for _name, _src in _GRAFT_SOURCES.items():
    (_G_DIR / _name).write_text(_src, encoding="utf-8")
if str(_G_DIR) not in sys.path:
    sys.path.insert(0, str(_G_DIR))
_installed = {}
for _mod_name in ['graft_pipeline']:
    _m = _il.import_module(_mod_name)
    _st = _m.install()
    _installed[_mod_name] = _st
    assert _st.endswith(": OK"), _st
print("[ab] grafts installed:", _installed, flush=True)


In [ ]:
# ==== two-phase run: stock_a -> stock_b (one boot, same server) ====
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))
assert not TRUE_SUBMISSION, "A/B smoke kernel must never run as a submission"

_ENV_DIR = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
PHASES = [('stock_a', {'TP_ENABLE': '0', 'TP2_ENABLE': '0', 'TP4_ENABLE': '0', 'TP5_ENABLE': '0', 'TP6_ENABLE': '0', 'TP7_ENABLE': '0', 'TP8_ENABLE': '0', 'TP9_ENABLE': '0', 'TP10_ENABLE': '0'}), ('stock_b', {'TP_ENABLE': '0', 'TP2_ENABLE': '0', 'TP4_ENABLE': '0', 'TP5_ENABLE': '0', 'TP6_ENABLE': '0', 'TP7_ENABLE': '0', 'TP8_ENABLE': '0', 'TP9_ENABLE': '0', 'TP10_ENABLE': '0'})]
AB_RESULTS = {}
_PHASE_BUDGET_S = 4.0 * 3600
_METRICS_URL = os.environ.get("LOCAL_ANALYZER_BASE_URL", "http://127.0.0.1:1234/v1").rstrip("/")
_METRICS_URL = _METRICS_URL[:-3] + "/metrics" if _METRICS_URL.endswith("/v1") else _METRICS_URL + "/metrics"
_METRIC_KEYS = ("prefix_cache_hits", "prefix_cache_queries", "prompt_tokens", "generation_tokens",
                "request_success", "num_preemptions", "request_prefill_time_seconds_sum",
                "request_decode_time_seconds_sum", "e2e_request_latency_seconds_sum",
                "e2e_request_latency_seconds_count")


def _vllm_counters() -> dict:
    """Best-effort snapshot of vLLM's Prometheus counters (never raises)."""
    out = {}
    try:
        import urllib.request

        with urllib.request.urlopen(_METRICS_URL, timeout=10) as resp:
            text = resp.read().decode("utf-8", "replace")
        for line in text.splitlines():
            if line.startswith("#") or not line.startswith("vllm:"):
                continue
            name, _, val = line.partition(" ")
            base = name.split("{", 1)[0][len("vllm:"):]
            for key in _METRIC_KEYS:
                if base == key or base == key + "_total":
                    try:
                        out[key] = out.get(key, 0.0) + float(val)
                    except ValueError:
                        pass
    except Exception as _exc:  # noqa: BLE001
        out["error"] = repr(_exc)[:200]
    return out


def _counter_delta(before: dict, after: dict) -> dict:
    d = {k: round(after[k] - before[k], 3) for k in after if k in before and k != "error"}
    if d.get("prefix_cache_queries"):
        d["prefix_hit_rate"] = round(d.get("prefix_cache_hits", 0.0) / d["prefix_cache_queries"], 4)
    for src in (before, after):
        if "error" in src:
            d["error"] = src["error"]
    return d


def _graft_counters() -> dict:
    """TP9 telemetry counters (attests the arm fired / stayed off). Never raises."""
    try:
        import sys as _sys

        st = _sys.modules["graft_pipeline"].status()
        return {k: st.get(k) for k in ("enabled", "resumes_injected",
                                       "perturbations_injected", "retries_used")}
    except Exception as _exc:  # noqa: BLE001
        return {"error": repr(_exc)[:200]}


def _server_pid() -> str:
    try:
        return (WORKING_DIR / "vllm-openai-server.pid").read_text(encoding="utf-8").strip()
    except Exception as _exc:  # noqa: BLE001
        return "unreadable: " + repr(_exc)[:120]

_v31_start_watchdog(bm.solver)
try:
    for _phase_name, _phase_env in PHASES:
        for _k, _v in _phase_env.items():
            os.environ[_k] = _v
        bm.game_runs = []
        bm.games = _offline_games(_ENV_DIR)
        bm.n_passes = 1
        bm.game_weights = None
        bm.label = "v22-ab-" + _phase_name
        _soft_end = datetime.now() + timedelta(seconds=_PHASE_BUDGET_S)
        _m_before = _vllm_counters()
        _g_before = _graft_counters()
        _pid_before = _server_pid()
        _t_start = datetime.utcnow().isoformat()
        print(f"=== PHASE {_phase_name}: {len(bm.games)} games env={_phase_env} "
              f"soft_end={_soft_end} vllm_before={_m_before} graft_before={_g_before} "
              f"server_pid={_pid_before} ===", flush=True)
        try:
            await bm.run(soft_end_time=_soft_end, runtime_environment=target,
                         minimal_diagnostics=False)
        except Exception as _exc:  # noqa: BLE001
            import traceback

            print(f"PHASE {_phase_name} RAISED {type(_exc).__name__}: {_exc}", flush=True)
            traceback.print_exc()
        games = []
        for game_run in list(bm.game_runs):
            apl = list(game_run.actions_per_level or [])
            games.append({
                "game_id": game_run.game_id,
                "state": str(game_run.state),
                "levels_completed": game_run.levels_completed,
                "number_of_levels": game_run.number_of_levels,
                "final_score": game_run.final_score,
                "actions": sum(apl) if apl else len(game_run.history),
                "actions_per_level": apl,
                "wallclock_s": game_run.final_wallclock_seconds,
            })
        n = max(1, len(games))
        _m_after = _vllm_counters()
        _g_after = _graft_counters()
        _graft_delta = {k: (_g_after.get(k, 0) or 0) - (_g_before.get(k, 0) or 0)
                        for k in ("resumes_injected", "perturbations_injected", "retries_used")
                        if isinstance(_g_after.get(k), int) and isinstance(_g_before.get(k), int)}
        _graft_delta["enabled_at_start"] = _g_before.get("enabled")
        _graft_delta["enabled_at_end"] = _g_after.get("enabled")
        AB_RESULTS[_phase_name] = {
            "graft": _graft_delta,
            "server_pid_start": _pid_before,
            "server_pid_end": _server_pid(),
            "games": games,
            "n": len(games),
            "phase_index": len(AB_RESULTS),
            "env": dict(_phase_env),
            "started_utc": _t_start,
            "ended_utc": datetime.utcnow().isoformat(),
            "vllm": _counter_delta(_m_before, _m_after),
            "mean_score": round(sum(g["final_score"] for g in games) / n, 3),
            "mean_levels": round(sum(g["levels_completed"] for g in games) / n, 3),
            "zero_level": sum(1 for g in games if not g["levels_completed"]),
            "total_actions": sum(g["actions"] for g in games),
        }
        (WORKING_DIR / "ab_results.json").write_text(json.dumps(AB_RESULTS, indent=1))
        print(f"=== PHASE READ {_phase_name}: mean_score={AB_RESULTS[_phase_name]['mean_score']} "
              f"mean_levels={AB_RESULTS[_phase_name]['mean_levels']} "
              f"zero={AB_RESULTS[_phase_name]['zero_level']}/{n} "
              f"actions={AB_RESULTS[_phase_name]['total_actions']} "
              f"vllm={AB_RESULTS[_phase_name]['vllm']} graft={_graft_delta} "
              f"pid={_pid_before}->{AB_RESULTS[_phase_name]['server_pid_end']} ===", flush=True)
finally:
    _v31_stop_watchdog()
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}", flush=True)
        subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())

print("\n==== SUMMARY (stock_a -> stock_b) ====")
for _p, _r in AB_RESULTS.items():
    print(f"{_p:8s} mean_score={_r['mean_score']} mean_levels={_r['mean_levels']} "
          f"zero={_r['zero_level']}/{_r['n']} actions={_r['total_actions']} vllm={_r['vllm']} "
          f"graft={_r['graft']} pid={_r['server_pid_start']}->{_r['server_pid_end']}")
